In [2]:
import subprocess
import sys
subprocess.check_call([sys.executable,'-m','pip','install',
    'qiskit==2.2.3','qiskit-machine-learning==0.9.0','numpy==2.2.6',
    'scipy==1.15.3','scikit-learn==1.6.1','pandas==2.2.3',
    'matplotlib==3.10.1','flask==3.1.0'])

0

In [7]:
import time
from pathlib import Path
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from qiskit.circuit.library import z_feature_map, zz_feature_map, pauli_feature_map
from qiskit.circuit.library import real_amplitudes, efficient_su2, n_local, pauli_two_design
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.primitives import QMLSampler, QMLEstimator
from qiskit_machine_learning.neural_networks import SamplerQNN, EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import VQC

def parity(integer):
    return int(integer).bit_count() % 2

def circuits(n, fmap_name='Z', ansatz_name='RealAmplitudes', reps=1):
    if fmap_name == 'Z':
        fmap = z_feature_map(n, reps=1)
    elif fmap_name == 'ZZ':
        fmap = zz_feature_map(n, reps=1, entanglement='linear')
    elif fmap_name == 'Pauli':
        fmap = pauli_feature_map(n, reps=1, paulis=['Z','XX'], entanglement='linear')
    else:
        raise ValueError(fmap_name)
    if ansatz_name == 'RealAmplitudes':
        ansatz = real_amplitudes(n, reps=reps, entanglement='linear')
    elif ansatz_name == 'EfficientSU2':
        ansatz = efficient_su2(n, reps=reps, entanglement='linear')
    elif ansatz_name == 'TwoLocal':
        ansatz = n_local(n, rotation_blocks=['ry','rz'], entanglement_blocks='cz', reps=reps, entanglement='linear')
    elif ansatz_name == 'PauliTwoDesign':
        ansatz = pauli_two_design(n, reps=reps, seed=33)
    else:
        raise ValueError(ansatz_name)
    return fmap, ansatz

def fit_model(kind, train_x, train_y, fmap_name='Z', ansatz_name='RealAmplitudes', maxiter=40, reps=1):
    fmap, ansatz = circuits(train_x.shape[1], fmap_name, ansatz_name, reps)
    initial = np.linspace(0.1,0.7,ansatz.num_parameters)
    history = []
    def optimizer(fun, x0, jac=None, bounds=None):
        def tracked(weights):
            value = float(fun(weights))
            history.append(value)
            return value
        return minimize(tracked, x0, method='COBYLA', options={'maxiter':maxiter, 'rhobeg':0.3})
    started = time.perf_counter()
    if kind == 'VQC':
        model = VQC(feature_map=fmap, ansatz=ansatz, sampler=QMLSampler(shots=None),
                    interpret=parity, output_shape=2, initial_point=initial, optimizer=optimizer)
        model.fit(train_x, train_y)
        def predict_probability(x):
            return np.asarray(model.neural_network.forward(x, model.weights))[:,1]
        weights = model.weights
    else:
        circuit = fmap.compose(ansatz)
        if kind == 'SQNN':
            qnn = SamplerQNN(circuit=circuit, sampler=QMLSampler(shots=None),
                             input_params=list(fmap.parameters), weight_params=list(ansatz.parameters),
                             interpret=parity, output_shape=2)
            def probability(x,w):
                return np.asarray(qnn.forward(x,w))[:,1]
            start = initial
        elif kind == 'EQNN':
            qnn = EstimatorQNN(circuit=circuit, estimator=QMLEstimator(default_precision=0),
                              observables=SparsePauliOp('Z'*train_x.shape[1]),
                              input_params=list(fmap.parameters), weight_params=list(ansatz.parameters),
                              default_precision=0)
            def probability(x,w):
                expectation = np.asarray(qnn.forward(x,w[:-2])).ravel()
                # 마지막 두 값은 고전 출력층의 가중치와 편향입니다.
                score = np.clip(w[-2]*expectation+w[-1],-40,40)
                return 1/(1+np.exp(-score))
            start = np.r_[initial,-1.0,0.0]
        else:
            raise ValueError(kind)
        def loss(w):
            p1 = np.clip(probability(train_x,w),1e-10,1-1e-10)
            value = float(-np.mean(train_y*np.log2(p1)+(1-train_y)*np.log2(1-p1)))
            return value
        result = optimizer(loss,start)
        weights = result.x
        def predict_probability(x):
            return probability(x,weights)
        model = None
    print(kind, fmap_name, ansatz_name, '학습 평가 수:',len(history),
          '최초/마지막 기록 손실:',round(history[0],6),round(history[-1],6))
    return predict_probability, model, {'parameters':len(weights),'depth':fmap.compose(ansatz).depth(),
                                       'seconds':time.perf_counter()-started,'history':history}

def report(name,y,p1):
    p1 = np.asarray(p1).ravel()
    assert p1.shape == y.shape and np.isfinite(p1).all()
    assert np.all((p1>=-1e-8)&(p1<=1+1e-8))
    predicted = (p1>=0.5).astype(int)
    result = {'model':name, 'accuracy':accuracy_score(y,predicted),
              'precision':precision_score(y,predicted,zero_division=0),
              'recall':recall_score(y,predicted,zero_division=0),
              'f1':f1_score(y,predicted,zero_division=0)}
    print({k:round(float(v),4) if k!='model' else v for k,v in result.items()})
    print('혼동행렬 [[TN,FP],[FN,TP]]:',confusion_matrix(y,predicted,labels=[0,1]).tolist())
    return result

def split_fixed(X,y):
    # 각 클래스에서 원래 행 순서의 네 번째마다 시험 표본으로 사용합니다.
    test = np.concatenate([np.flatnonzero(y==c)[3::4] for c in [0,1]])
    train = np.setdiff1d(np.arange(len(y)),test)
    assert not np.intersect1d(train,test).size
    assert len(np.unique(y[train]))==len(np.unique(y[test]))==2
    return train,test

print('버전:',{k:metadata.version(k) for k in ['qiskit','qiskit-machine-learning','numpy','scipy','scikit-learn']})

from urllib.request import urlretrieve
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
path=Path('creditcard.csv')
if not path.exists():
    urlretrieve('https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv',path)
df=pd.read_csv(path)
features=['V1','V2','V3','V4','V5','V6','V7']
assert set(features+['Class']).issubset(df.columns)
print('원본 행 수/사기 수:',len(df),int(df['Class'].sum()))
indices=[]
for label in [0,1]:
    candidates=df.index[df['Class']==label].to_numpy()
    indices.extend(candidates[np.linspace(0,len(candidates)-1,24,dtype=int)])
subset=df.loc[indices]
X=subset[features].to_numpy(float)
y=subset['Class'].to_numpy(int)
assert np.isfinite(X).all()
train,test=split_fixed(X,y)
standard=StandardScaler().fit(X[train])
pca=PCA(n_components=4,svd_solver='full').fit(standard.transform(X[train]))
train_pca=pca.transform(standard.transform(X[train]))
test_pca=pca.transform(standard.transform(X[test]))
scaler=MinMaxScaler(clip=True).fit(train_pca)
train_x,test_x=scaler.transform(train_pca),scaler.transform(test_pca)
print('학습/시험 크기:',train_x.shape,test_x.shape)
print('학습/시험 라벨 수:',np.bincount(y[train]).tolist(),np.bincount(y[test]).tolist())
print('PCA 누적 설명분산:',round(float(pca.explained_variance_ratio_.sum()),4))
results=[]
for kind in ['VQC','SQNN','EQNN']:
    predict,model,info=fit_model(kind,train_x,y[train],maxiter=40)
    results.append(report(kind,y[test],predict(test_x)))
    print('파라미터/깊이/시간:',info['parameters'],info['depth'],round(info['seconds'],3))
baseline=LogisticRegression(max_iter=300).fit(train_x,y[train])
results.append(report('LogisticRegression',y[test],baseline.predict_proba(test_x)[:,1]))
pd.DataFrame(results).to_csv('fraud_real_results.csv',index=False)
fig,ax=plt.subplots(figsize=(7,4))
ax.bar([r['model'] for r in results],[r['f1'] for r in results])
ax.set(ylabel='F1',ylim=(0,1))
fig.tight_layout();fig.savefig('fraud_real_scores.png');plt.close(fig)

버전: {'qiskit': '2.2.3', 'qiskit-machine-learning': '0.9.0', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit-learn': '1.6.1'}


원본 행 수/사기 수: 284807 492
학습/시험 크기: (36, 4) (12, 4)
학습/시험 라벨 수: [18, 18] [6, 6]
PCA 누적 설명분산: 0.9447


VQC Z RealAmplitudes 학습 평가 수: 40 최초/마지막 기록 손실: 1.004401 0.820852
{'model': 'VQC', 'accuracy': 0.6667, 'precision': 1.0, 'recall': 0.3333, 'f1': 0.5}
혼동행렬 [[TN,FP],[FN,TP]]: [[6, 0], [4, 2]]
파라미터/깊이/시간: 8 7 10.721


SQNN Z RealAmplitudes 학습 평가 수: 40 최초/마지막 기록 손실: 1.004401 0.820852
{'model': 'SQNN', 'accuracy': 0.6667, 'precision': 1.0, 'recall': 0.3333, 'f1': 0.5}
혼동행렬 [[TN,FP],[FN,TP]]: [[6, 0], [4, 2]]
파라미터/깊이/시간: 8 7 7.949
EQNN Z RealAmplitudes 학습 평가 수: 40 최초/마지막 기록 손실: 0.999932 0.842712
{'model': 'EQNN', 'accuracy': 0.6667, 'precision': 1.0, 'recall': 0.3333, 'f1': 0.5}
혼동행렬 [[TN,FP],[FN,TP]]: [[6, 0], [4, 2]]
파라미터/깊이/시간: 10 7 3.004
{'model': 'LogisticRegression', 'accuracy': 0.8333, 'precision': 1.0, 'recall': 0.6667, 'f1': 0.8}
혼동행렬 [[TN,FP],[FN,TP]]: [[6, 0], [2, 4]]


이 실행에서 VQC·SQNN·EQNN의 F1은 각각 0.5이고 로지스틱 회귀의 F1은 0.8입니다. 세 양자 모델의 혼동행렬 [[6,0],[4,2]]는 정상 6건을 맞히고 사기 6건 중 2건을 발견했음을 뜻합니다. 정밀도 1.0과 재현율 0.3333을 함께 읽어야 미탐 4건을 확인할 수 있습니다. VQC와 SQNN은 같은 조건의 확률 모델이므로 손실과 예측이 일치합니다.

학습·시험 표본 수와 최대 평가 횟수가 논문과 다르므로 출력값은 이 실습 조건의 결과입니다. 클래스 균형을 바꾼 데이터의 정밀도는 운영 환경의 정밀도와 함께 해석할 별도 지표입니다.